In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

batch_size = 50

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [ ]:
# 2. Define Network Parameters
Din = 3 * 32 * 32  # Input size (flattened CIFAR-10 image size)
K = 10  # Output size (number of classes in CIFAR-10)
std = 1e-5
hidden_units = 100

# Input to hidden layer weights and biases
w1 = torch.randn(Din, hidden_units) * std
b1 = torch.zeros(hidden_units)

# Hidden to output layer weights and biases
w2 = torch.randn(hidden_units, K) * std
b2 = torch.zeros(K)

# Hyperparameters
iterations = 10
lr = 1e-2  # Learning rate (Increased)
lr_decay = 0.99  # Learning rate decay
reg = 0.0  # Regularization
loss_history = []

In [ ]:
# sigmoid function
def sigmoid(x):
    return 1 / (1 + torch.exp(-x))

# CrossEntropy Loss
def CrossEntropyLoss(y_pred, y_true):
    smooth = 1e-9
    loss = -torch.sum(y_true * torch.log(y_pred + smooth)) / y_true.shape[0]
    return loss

In [ ]:
# 3. Training Loop
for t in range(iterations):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # Get inputs and labels
        inputs, labels = data
        Ntr = inputs.shape[0]  # Batch size
        x_train = inputs.view(Ntr, -1)  # Flatten input to (Ntr, Din)
        y_train_onehot = nn.functional.one_hot(labels, K).float()  # Convert labels to one-hot encoding

        # Forward pass with middle layer
        hidden = sigmoid(x_train.mm(w1) + b1)  # Middle layer with sigmoid activation
        y_pred = torch.softmax(hidden.mm(w2) + b2, dim=1) # Output layer activation

        # Loss calculation (CrossEntropy loss with regularization for both layers)
        loss = CrossEntropyLoss(y_pred, y_train_onehot)  # Cross-entropy loss
        loss += reg * (torch.sum(w1 ** 2) + torch.sum(w2 ** 2))  # Also add regularization term
        loss_history.append(loss.item())
        running_loss += loss.item()

        # Backpropagation
        dy_pred = (y_pred - y_train_onehot) # Loss derivative
        dw2 = hidden.t().mm(dy_pred) + reg * w2
        db2 = dy_pred.sum(dim=0)

        # Backpropagation to hidden layer
        dhidden = dy_pred.mm(w2.t()) * (hidden * (1 - hidden))  # Derivative of sigmoid
        dw1 = x_train.t().mm(dhidden) + reg * w1# 4. Plotting the Loss History
        db1 = dhidden.sum(dim=0)

        # Parameter update
        w1 -= lr * dw1
        b1 -= lr * db1
        w2 -= lr * dw2
        b2 -= lr * db2

    # Print loss for every epoch
    if t % 1 == 0:
        print(f"Epoch {t + 1} / {iterations}, Loss: {running_loss / len(trainloader)}")

    # Learning rate decay
    lr *= lr_decay

In [ ]:
plt.figure()
plt.plot(loss_history, color='blue')
plt.title("Loss History")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.show()

In [ ]:
# 5. Calculate Accuracy on Training Set
correct_train = 0
total_train = 0
with torch.no_grad():
    for data in trainloader:
        inputs, labels = data
        Ntr = inputs.shape[0]
        x_train = inputs.view(Ntr, -1)
        y_train_onehot = nn.functional.one_hot(labels, K).float()

        # Forward pass
        hidden_layer = sigmoid(x_train.mm(w1) + b1)  # Middle layer with sigmoid activation
        y_train_pred = hidden_layer.mm(w2) + b2  # Output layer activation
        predicted_train = torch.argmax(y_train_pred, dim=1)
        total_train += labels.size(0)
        correct_train += (predicted_train == labels).sum().item()

train_acc = 100 * correct_train / total_train
print(f"Training accuracy: {train_acc:.2f}%")

In [ ]:
# 6. Calculate Accuracy on Test Set
correct_test = 0
total_test = 0
with torch.no_grad():
    for data in testloader:
        inputs, labels = data
        Nte = inputs.shape[0]
        x_test = inputs.view(Nte, -1)
        y_test_onehot = nn.functional.one_hot(labels, K).float()

        # Forward pass
        hidden_layer = sigmoid(x_test.mm(w1) + b1)  # Middle layer with sigmoid activation
        y_test_pred = hidden_layer.mm(w2) + b2  # Output layer activation
        predicted_test = torch.argmax(y_test_pred, dim=1)
        total_test += labels.size(0)
        correct_test += (predicted_test == labels).sum().item()

test_acc = 100 * correct_test / total_test
print(f"Test accuracy: {test_acc:.2f}%")